# 04 — Context Relevance, By Hand

Notebook 03 asked "did retrieval find the right document." This one asks a narrower, easy-to-overlook question: retrieval already succeeded -- the right document is sitting right here -- so **how much of what actually reached the model was useful, versus noise it had to read past?**

These are genuinely different failure categories, even though they can look identical from the outside. A document being "the right one" says nothing about whether the specific passage handed to the model was clean or cluttered.


In [ ]:
import json
import os

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)

cuad_examples = examples[0:10]  # single-fact + numeric extraction, both CUAD


## Worked example 1 — the answer is one paragraph in five

Real contract text almost never arrives as a single clean sentence. Read the whole window, not just the gold answer, before judging anything.


In [ ]:
ex = examples[1]
print("Question:", ex["question"])
print()
print(ex["context_text"])
print()
print("Gold answer:", ex["gold_answer"])


**Judgment:**
- **Relevant:** exactly one sentence -- "This Agreement was entered into in the State of Florida, and its validity... shall be governed by the laws and judicial decisions of the State of Florida..."
- **Irrelevant / noise:** the waiver clause, the severability clause ("if any provision is found invalid..."), and the force majeure clause (act of God, war, fire, earthquake...) are all real contract boilerplate, unrelated to governing law, and take up more of the window than the answer itself.
- **Missing:** nothing -- the full governing-law sentence is intact, not cut off at either edge.

This context is a genuinely good retrieval result carrying a genuinely noisy payload: about one-fifth of the window is the answer, the rest is boilerplate the model has to read past. A model that answers this correctly (which the system did, back in notebook 01) is filtering noise well. A model that answered incorrectly here couldn't blame retrieval -- the signal was present, just diluted.


## Worked example 2 — when the window cuts something important

Not every context is clean noise-around-a-full-answer. Sometimes the window itself is the problem.


In [ ]:
ex = examples[5]
print("Question:", ex["question"])
print()
print(ex["context_text"])
print()
print("Gold answer:", ex["gold_answer"])


**Judgment:**
- **Relevant:** the "LIMITATION OF LIABILITY" section -- two full sentences stating no liability for lost profits/consequential damages, and a liability cap of one month's fees plus prepaid fees. Complete and unambiguous.
- **Irrelevant / noise:** the renewal-notice and termination clauses above it, and the start of an "INDEMNIFICATION" section below it -- a different clause entirely, cut off mid-sentence at the very end of the window ("and wi...").
- **Missing:** nothing from the actual answer -- but that trailing cut-off indemnification fragment is worth noticing anyway. It's irrelevant to *this* question, but if a later question asked about indemnification instead, this exact window would badly fail worked example 1's "was anything cut off" check.

Same lesson as example 1, with a twist: noise isn't always harmless filler. Here, part of the noise is a different real clause, sliced in half by where the window happened to end. A system asked about indemnification, handed this exact chunk, would have every reason to answer badly -- and it wouldn't be a generation failure. It would be this chunk boundary.


## Your turn: the remaining 8 CUAD examples

Same three questions for each: what's relevant, what's noise, what (if anything) seems to be missing or cut off at the edges.


In [ ]:
remaining = [i for i in range(10) if i not in (1, 5)]

for i in remaining:
    ex = examples[i]
    print(f"[{i}] {ex['question']}")
    print(ex["context_text"])
    print("Gold:", ex["gold_answer"][:200])
    print("-" * 60)


In [ ]:
context_relevance_notes = {
    1: {"relevant": "The Florida governing-law sentence.", "irrelevant": "Waiver, severability, force majeure clauses.", "missing": "Nothing."},
    5: {"relevant": "The two LIMITATION OF LIABILITY sentences.", "irrelevant": "Renewal/termination clause above; indemnification clause below.", "missing": "Nothing for this question, but the indemnification section is cut off mid-sentence."},
    # Fill in for the remaining 8 indices above:
    # 0: {"relevant": "", "irrelevant": "", "missing": ""},
}

print(f"Notes written for {len(context_relevance_notes)}/10")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Context relevance | How much of a retrieved chunk is actually useful, versus noise the model has to filter past |
| Chunk boundary artifact | A different, unrelated clause cut off mid-sentence at the edge of a retrieved window |

The core distinction to carry forward: retrieval can succeed completely (the right document, even the right general area of it) while context quality still varies wildly -- from "one clean answer buried in boilerplate" to "the answer plus a real, different clause sliced in half at the edge." Both of your worked examples had the right answer intact. Not every real chunk will.

**Next up:** notebook `05` moves from "was the context good" to "did the answer actually stick to it" -- faithfulness, including the case where an answer sounds completely reasonable and still isn't supported by anything in front of the model.

**Exercises:**

- Finish `context_relevance_notes` for all 8 remaining examples.
- Find the example (if any) among the 10 where the irrelevant content is the *largest* share of the window. Does that example's system answer (check `examples[i]["system_answer"]`) show any sign of the model getting distracted by it?
- For example 5, imagine a follow-up question specifically about indemnification terms. Using only what's visible in that exact context window, could it be answered completely? What's missing?
